In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.metrics import mean_squared_error

In [ ]:

# ============================================================
# Neural Network Forecasting
# Point Forecast + Residual-based PI + Conformal Calibration
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.metrics import mean_squared_error


# ============================================================
# SETTINGS
# ============================================================

DATA_FILE = "100_clients_sample.parquet"

# 100-client experiment:
N_SELECTED_CLIENTS = 20

# For 1000-client experiment, use:
# DATA_FILE = "1000_clients_sample.parquet"
# N_SELECTED_CLIENTS = 100

HISTORY = 100
HORIZON = 28

TRAIN_RATIO = 0.60
CAL_RATIO = 0.20
TEST_RATIO = 0.20

EPOCHS = 50
LEARNING_RATE = 1e-3

ALPHA = 0.10
Z_VALUE = 1.645

DROPOUT = 0.20

RANDOM_SEED = 42


# ============================================================
# REPRODUCIBILITY
# ============================================================

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ============================================================
# LOAD DATA
# ============================================================

data = pd.read_parquet(DATA_FILE)

print("Data shape:", data.shape)
print(data.head())


# ============================================================
# CHECK REQUIRED COLUMNS
# ============================================================

required_columns = {
    "eup_grid_id",
    "date",
    "balance_in_euro"
}

missing_columns = (
    required_columns
    - set(data.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ============================================================
# WINKLER SCORE
# ============================================================

def winkler_score(
    y_true,
    lower,
    upper,
    alpha=0.10
):

    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    y = y_true.flatten()
    l = lower.flatten()
    u = upper.flatten()

    width = u - l

    score = width.copy()

    below = y < l
    above = y > u

    score[below] += (
        2 / alpha
    ) * (
        l[below] - y[below]
    )

    score[above] += (
        2 / alpha
    ) * (
        y[above] - u[above]
    )

    return np.mean(score)


# ============================================================
# CREATE SLIDING WINDOWS
# ============================================================

def make_windows(
    y,
    history=100,
    horizon=28
):

    X = []
    Y = []

    for i in range(
        len(y)
        - history
        - horizon
        + 1
    ):

        X.append(
            y[
                i:
                i + history
            ]
        )

        Y.append(
            y[
                i + history:
                i + history + horizon
            ]
        )

    return (
        np.asarray(X),
        np.asarray(Y)
    )


# ============================================================
# NEURAL NETWORK
# ============================================================

class ForecastNet(nn.Module):

    def __init__(
        self,
        history=100,
        horizon=28,
        dropout=0.20
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                history,
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                256,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                horizon
            )
        )

    def forward(
        self,
        x
    ):

        return self.net(x)


# ============================================================
# SINGLE-CLIENT PIPELINE
# ============================================================

def run_nn_pipeline(
    series,
    dates
):

    history = HISTORY
    horizon = HORIZON

    series = np.asarray(
        series,
        dtype=float
    )

    dates = pd.to_datetime(
        dates
    )


    # --------------------------------------------------------
    # CHECK LOG1P REQUIREMENT
    # --------------------------------------------------------

    if np.min(series) <= -1:

        raise ValueError(
            "balance_in_euro contains values <= -1. "
            "np.log1p() cannot be used safely. "
            "Use the same alternative transformation "
            "for AutoARIMA, DeepAR, and NN."
        )


    # ========================================================
    # LOG TRANSFORMATION
    # ========================================================

    y_log = np.log1p(
        series
    )


    # ========================================================
    # SLIDING WINDOWS
    # ========================================================

    X, Y = make_windows(
        y_log,
        history=history,
        horizon=horizon
    )

    if len(X) < 50:

        print(
            "Not enough windows:",
            len(X)
        )

        return None


    # ========================================================
    # CHRONOLOGICAL 60 / 20 / 20 SPLIT
    # ========================================================

    n = len(X)

    train_end = int(
        n * TRAIN_RATIO
    )

    cal_end = int(
        n * (
            TRAIN_RATIO
            + CAL_RATIO
        )
    )

    X_train = X[
        :train_end
    ]

    Y_train = Y[
        :train_end
    ]

    X_cal = X[
        train_end:
        cal_end
    ]

    Y_cal = Y[
        train_end:
        cal_end
    ]

    X_test = X[
        cal_end:
    ]

    Y_test = Y[
        cal_end:
    ]

    if (
        len(X_train) == 0
        or len(X_cal) == 0
        or len(X_test) == 0
    ):

        return None


    # ========================================================
    # STANDARDIZATION
    # TRAINING DATA ONLY
    # ========================================================

    train_mean = np.mean(
        X_train
    )

    train_std = (
        np.std(
            X_train
        )
        + 1e-6
    )

    X_train_s = (
        X_train
        - train_mean
    ) / train_std

    X_cal_s = (
        X_cal
        - train_mean
    ) / train_std

    X_test_s = (
        X_test
        - train_mean
    ) / train_std

    Y_train_s = (
        Y_train
        - train_mean
    ) / train_std


    # ========================================================
    # CONVERT TO TENSORS
    # ========================================================

    Xt = torch.tensor(
        X_train_s,
        dtype=torch.float32
    ).to(device)

    Yt = torch.tensor(
        Y_train_s,
        dtype=torch.float32
    ).to(device)

    Xc = torch.tensor(
        X_cal_s,
        dtype=torch.float32
    ).to(device)

    Xs = torch.tensor(
        X_test_s,
        dtype=torch.float32
    ).to(device)


    # ========================================================
    # INITIALIZE MODEL
    # ========================================================

    model = ForecastNet(
        history=history,
        horizon=horizon,
        dropout=DROPOUT
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    criterion = nn.MSELoss()


    # ========================================================
    # TRAIN MODEL
    # ========================================================

    for epoch in range(
        EPOCHS
    ):

        model.train()

        optimizer.zero_grad()

        pred = model(
            Xt
        )

        loss = criterion(
            pred,
            Yt
        )

        loss.backward()

        optimizer.step()


    # ========================================================
    # PREDICTIONS
    # ========================================================

    model.eval()

    with torch.no_grad():

        train_pred_s = (
            model(Xt)
            .cpu()
            .numpy()
        )

        cal_pred_s = (
            model(Xc)
            .cpu()
            .numpy()
        )

        test_pred_s = (
            model(Xs)
            .cpu()
            .numpy()
        )


    # ========================================================
    # BACK TO LOG SCALE
    #
    # IMPORTANT:
    # Conformal calibration is performed BEFORE expm1,
    # so it is performed on the transformed scale.
    # ========================================================

    train_pred_log = (
        train_pred_s
        * train_std
        + train_mean
    )

    cal_pred_log = (
        cal_pred_s
        * train_std
        + train_mean
    )

    test_pred_log = (
        test_pred_s
        * train_std
        + train_mean
    )


    # ========================================================
    # TRAINING RESIDUAL UNCERTAINTY
    #
    # One residual standard deviation for each forecast
    # horizon:
    #
    # sigma_1, sigma_2, ..., sigma_28
    # ========================================================

    train_residuals_log = (
        Y_train
        - train_pred_log
    )

    residual_std_h = (
        np.std(
            train_residuals_log,
            axis=0
        )
        + 1e-8
    )

    # Shape should be (28,)
    assert (
        residual_std_h.shape[0]
        == horizon
    )


    # ========================================================
    # INITIAL 90% PREDICTION INTERVALS
    # ON LOG SCALE
    # ========================================================

    lower_cal_log = (
        cal_pred_log
        - Z_VALUE
        * residual_std_h[
            None, :
        ]
    )

    upper_cal_log = (
        cal_pred_log
        + Z_VALUE
        * residual_std_h[
            None, :
        ]
    )

    lower_test_log = (
        test_pred_log
        - Z_VALUE
        * residual_std_h[
            None, :
        ]
    )

    upper_test_log = (
        test_pred_log
        + Z_VALUE
        * residual_std_h[
            None, :
        ]
    )


    # ========================================================
    # CONFORMAL CALIBRATION
    # ========================================================

    scores = np.maximum(
        lower_cal_log
        - Y_cal,

        Y_cal
        - upper_cal_log
    )


    # --------------------------------------------------------
    # HORIZON-SPECIFIC CONFORMAL CORRECTIONS
    #
    # q_1, q_2, ..., q_28
    # --------------------------------------------------------

    qhat = []

    for h in range(
        horizon
    ):

        q = np.quantile(
            scores[:, h],
            1 - ALPHA,
            method="higher"
        )

        # Do not shrink the initial interval
        qhat.append(
            max(
                0.0,
                float(q)
            )
        )

    qhat = np.asarray(
        qhat
    )


    # ========================================================
    # FINAL CONFORMAL INTERVALS
    # ON LOG SCALE
    # ========================================================

    lower_conf_log = (
        lower_test_log
        - qhat[
            None, :
        ]
    )

    upper_conf_log = (
        upper_test_log
        + qhat[
            None, :
        ]
    )


    # ========================================================
    # INVERSE TRANSFORMATION TO EUR
    # ========================================================

    train_pred = np.expm1(
        train_pred_log
    )

    cal_pred = np.expm1(
        cal_pred_log
    )

    test_pred = np.expm1(
        test_pred_log
    )

    Y_train_real = np.expm1(
        Y_train
    )

    Y_cal_real = np.expm1(
        Y_cal
    )

    Y_test_real = np.expm1(
        Y_test
    )

    lower_conf = np.expm1(
        lower_conf_log
    )

    upper_conf = np.expm1(
        upper_conf_log
    )


    # ========================================================
    # TEST METRICS IN EUR
    # ========================================================

    rmse = np.sqrt(
        mean_squared_error(
            Y_test_real.flatten(),
            test_pred.flatten()
        )
    )

    coverage = np.mean(
        (
            Y_test_real
            >= lower_conf
        )
        &
        (
            Y_test_real
            <= upper_conf
        )
    )

    winkler = winkler_score(
        Y_test_real,
        lower_conf,
        upper_conf,
        alpha=ALPHA
    )


    # ========================================================
    # STORE DATES FOR LAST TEST WINDOW
    # ========================================================

    # The final generated window corresponds to the
    # final 28 observations in the series.
    forecast_dates = dates[
        -horizon:
    ]

    history_dates = dates[
        -(history + horizon):
        -horizon
    ]

    history_values = series[
        -(history + horizon):
        -horizon
    ]


    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return {

        "test_rmse":
            rmse,

        "test_coverage":
            coverage,

        "test_winkler":
            winkler,

        "test_pred":
            test_pred,

        "y_test":
            Y_test_real,

        "lower_conf":
            lower_conf,

        "upper_conf":
            upper_conf,

        "qhat":
            qhat,

        "residual_std_h":
            residual_std_h,

        "history_dates":
            history_dates,

        "history_values":
            history_values,

        "forecast_dates":
            forecast_dates,

        "dates":
            dates,

        "series":
            series
    }


# ============================================================
# SELECT HIGH-VARIABILITY CLIENTS
# ============================================================

client_stats = (
    data
    .groupby(
        "eup_grid_id"
    )[
        "balance_in_euro"
    ]
    .agg(
        mean="mean",
        std="std"
    )
)


selected_clients = (
    client_stats
    .sort_values(
        "std",
        ascending=False
    )
    .head(
        N_SELECTED_CLIENTS
    )
    .index
    .tolist()
)


print(
    "\nSelected clients:",
    len(selected_clients)
)

print(
    selected_clients
)


# Optional:
# save the IDs so AutoARIMA / DeepAR / NN can use
# exactly the same clients.

pd.DataFrame(
    {
        "eup_grid_id":
            selected_clients
    }
).to_csv(
    "selected_clients.csv",
    index=False
)


# ============================================================
# RUN MODEL FOR ALL SELECTED CLIENTS
# ============================================================

all_results = {}

all_metrics = []


for client_id in selected_clients:

    try:

        subset = (
            data[
                data[
                    "eup_grid_id"
                ]
                == client_id
            ]
            .copy()
        )

        subset[
            "date"
        ] = pd.to_datetime(
            subset[
                "date"
            ]
        )

        subset = (
            subset
            .sort_values(
                "date"
            )
            .set_index(
                "date"
            )
            .asfreq(
                "D"
            )
        )


        # ----------------------------------------------------
        # MISSING VALUES
        # ----------------------------------------------------

        subset[
            "balance_in_euro"
        ] = (
            subset[
                "balance_in_euro"
            ]
            .ffill()
            .fillna(0)
        )


        series = (
            subset[
                "balance_in_euro"
            ]
            .astype(float)
            .values
        )


        # ----------------------------------------------------
        # RUN CLIENT PIPELINE
        # ----------------------------------------------------

        result = run_nn_pipeline(
            series,
            subset.index
        )

        if result is None:
            continue


        all_results[
            client_id
        ] = result


        # ----------------------------------------------------
        # SCALE FOR NORMALIZED METRICS
        # ----------------------------------------------------

        std_balance = (
            np.std(
                series
            )
            + 1e-8
        )


        all_metrics.append(
            {

                "client_id":
                    client_id,

                "rmse":
                    result[
                        "test_rmse"
                    ],

                "coverage":
                    result[
                        "test_coverage"
                    ],

                "winkler":
                    result[
                        "test_winkler"
                    ],

                "mean_balance":
                    np.mean(
                        series
                    ),

                "std_balance":
                    std_balance
            }
        )


        print(
            f"Done: {client_id} | "
            f"RMSE={result['test_rmse']:,.2f} | "
            f"Coverage={result['test_coverage']:.2%} | "
            f"Winkler={result['test_winkler']:,.2f}"
        )


    except Exception as e:

        print(
            f"Error for client {client_id}: "
            f"{e}"
        )


# ============================================================
# CLIENT-LEVEL METRICS
# ============================================================

metrics_df = pd.DataFrame(
    all_metrics
)


if len(
    metrics_df
) == 0:

    raise RuntimeError(
        "No clients were successfully evaluated."
    )


metrics_df[
    "nrmse"
] = (
    metrics_df[
        "rmse"
    ]
    /
    metrics_df[
        "std_balance"
    ]
)


metrics_df[
    "nwinkler"
] = (
    metrics_df[
        "winkler"
    ]
    /
    metrics_df[
        "std_balance"
    ]
)


print(
    "\nClient-level metrics:"
)

print(
    metrics_df.head()
)


# ============================================================
# RAW AVERAGES
# ============================================================

print(
    "\nAverage RMSE:",
    metrics_df[
        "rmse"
    ].mean()
)

print(
    "Average NRMSE:",
    metrics_df[
        "nrmse"
    ].mean()
)

print(
    "Average Coverage:",
    metrics_df[
        "coverage"
    ].mean()
)

print(
    "Average Winkler:",
    metrics_df[
        "winkler"
    ].mean()
)

print(
    "Average Normalized Winkler:",
    metrics_df[
        "nwinkler"
    ].mean()
)


# ============================================================
# AGGREGATE METRICS
# ============================================================

aggregate_metrics = {

    "n_clients":
        len(
            metrics_df
        ),

    "mean_nrmse":
        metrics_df[
            "nrmse"
        ].mean(),

    "median_nrmse":
        metrics_df[
            "nrmse"
        ].median(),

    "mean_coverage":
        metrics_df[
            "coverage"
        ].mean(),

    "median_coverage":
        metrics_df[
            "coverage"
        ].median(),

    "mean_nwinkler":
        metrics_df[
            "nwinkler"
        ].mean(),

    "median_nwinkler":
        metrics_df[
            "nwinkler"
        ].median()
}


print(
    "\nAggregate Neural Network Results"
)

print(
    "--------------------------------"
)


for key, value in (
    aggregate_metrics.items()
):

    if (
        "coverage"
        in key
    ):

        print(
            f"{key}: "
            f"{value:.2%}"
        )

    elif (
        key
        == "n_clients"
    ):

        print(
            f"{key}: "
            f"{value}"
        )

    else:

        print(
            f"{key}: "
            f"{value:.4f}"
        )


# ============================================================
# SELECT REPRESENTATIVE CLIENTS
# ============================================================

best_3 = (
    metrics_df
    .sort_values(
        "nrmse",
        ascending=True
    )
    .head(3)
)


worst_3 = (
    metrics_df
    .sort_values(
        "nrmse",
        ascending=False
    )
    .head(3)
)


print(
    "\nBest 3 clients by NRMSE:"
)

print(
    best_3[
        [
            "client_id",
            "nrmse",
            "coverage",
            "nwinkler"
        ]
    ]
)


print(
    "\nWorst 3 clients by NRMSE:"
)

print(
    worst_3[
        [
            "client_id",
            "nrmse",
            "coverage",
            "nwinkler"
        ]
    ]
)


# ============================================================
# PLOT FUNCTION
# ============================================================

def plot_client(
    result,
    client_id,
    metrics_row=None
):

    history_dates = (
        result[
            "history_dates"
        ]
    )

    history = (
        result[
            "history_values"
        ]
    )

    forecast_dates = (
        result[
            "forecast_dates"
        ]
    )


    # Last test window
    actual = (
        result[
            "y_test"
        ][-1]
    )

    forecast = (
        result[
            "test_pred"
        ][-1]
    )

    lower = (
        result[
            "lower_conf"
        ][-1]
    )

    upper = (
        result[
            "upper_conf"
        ][-1]
    )


    # ========================================================
    # WINDOW-SPECIFIC METRICS
    # ========================================================

    window_rmse = np.sqrt(
        mean_squared_error(
            actual,
            forecast
        )
    )

    window_coverage = np.mean(
        (
            actual
            >= lower
        )
        &
        (
            actual
            <= upper
        )
    )

    window_winkler = (
        winkler_score(
            actual,
            lower,
            upper,
            alpha=ALPHA
        )
    )


    # ========================================================
    # PLOT
    # ========================================================

    plt.figure(
        figsize=(15, 6)
    )


    # HISTORY
    plt.plot(
        history_dates,
        history,
        linewidth=2,
        label="History"
    )


    # ACTUAL
    plt.plot(
        forecast_dates,
        actual,
        linewidth=2,
        label="Actual"
    )


    # FORECAST
    plt.plot(
        forecast_dates,
        forecast,
        linewidth=2,
        label="Forecast"
    )


    # CONFORMAL PI
    plt.fill_between(
        forecast_dates,
        lower,
        upper,
        alpha=0.25,
        label="90% Conformal PI"
    )


    # FORECAST START
    plt.axvline(
        history_dates[-1],
        linestyle="--",
        linewidth=2,
        label="Forecast Start"
    )


    # ========================================================
    # TITLE
    # ========================================================

    title = (
        f"Neural Network Forecast\n"
        f"Window RMSE={window_rmse:,.0f} EUR | "
        f"Coverage={window_coverage:.1%} | "
        f"Winkler={window_winkler:,.0f}"
    )


    if (
        metrics_row
        is not None
    ):

        title += (
            "\n"
            f"Client NRMSE="
            f"{metrics_row['nrmse']:.3f} | "
            f"Client NWinkler="
            f"{metrics_row['nwinkler']:.3f}"
        )


    plt.title(
        title
    )

    plt.xlabel(
        "Date"
    )

    plt.ylabel(
        "Balance (€)"
    )

    plt.xticks(
        rotation=45
    )

    plt.grid(
        alpha=0.3
    )

    plt.legend()

    plt.tight_layout()

    plt.show()


# ============================================================
# PLOT BEST CLIENTS
# ============================================================

for client_id in (
    best_3[
        "client_id"
    ]
):

    row = (
        metrics_df[
            metrics_df[
                "client_id"
            ]
            == client_id
        ]
        .iloc[0]
    )

    plot_client(
        all_results[
            client_id
        ],
        client_id,
        metrics_row=row
    )


# ============================================================
# PLOT WORST CLIENTS
# ============================================================

for client_id in (
    worst_3[
        "client_id"
    ]
):

    row = (
        metrics_df[
            metrics_df[
                "client_id"
            ]
            == client_id
        ]
        .iloc[0]
    )

    plot_client(
        all_results[
            client_id
        ],
        client_id,
        metrics_row=row
    )


# ============================================================
# SAVE RESULTS
# ============================================================

metrics_df.to_csv(
    "nn_client_metrics.csv",
    index=False
)

pd.DataFrame(
    [
        aggregate_metrics
    ]
).to_csv(
    "nn_aggregate_metrics.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    "nn_client_metrics.csv"
)

print(
    "nn_aggregate_metrics.csv"
)